# Module 1 — Is this a job for AI, and what exactly is the system?

**The problem.** A sponsor says *"we need AI for customer service."* The most expensive mistake an AI architect can make is to start designing. The first job is to decide whether AI is even the right tool, and to turn a vague request into a bounded system with measurable success.

This notebook is a **general tutorial** — it teaches the feasibility framework and the problem-classification decision on illustrative examples. It is not tied to any one project. (The companion `scoping-document` **skill** is what you run on a *real* new project to produce its scoping doc.)

By the end you can: score a project on the five feasibility dimensions, spot the four hard disqualifiers, classify a problem onto the right approach, and decompose a business ask into the sub-tasks that actually need a model.


## 0. The example projects

Four deliberately different asks. We'll run every tool in this notebook against all four so the framework's verdicts are visible side by side.


In [1]:
from dataclasses import dataclass

@dataclass
class Project:
    name: str
    ask: str
    data_available: str      # strong | workable | weak | none
    task_specifiable: str    # can you write a correctness rubric? strong|workable|weak
    error_cost: str          # low | medium | high | catastrophic
    latency_need: str        # batch | interactive | realtime_hard
    hitl_possible: str       # yes | limited | no
    deterministic: bool      # is the core actually a fixed rule?
    regulated_forbidden: bool

PROJECTS = [
    Project("Invoice totals", "Automate adding up invoice line items",
            "strong","strong","high","interactive", "yes", deterministic=True, regulated_forbidden=False),
    Project("Support triage", "Route and draft replies for customer support tickets",
            "workable","workable","medium","interactive","yes", deterministic=False, regulated_forbidden=False),
    Project("Fraud blocking", "Auto-decline transactions the model thinks are fraud, no human",
            "strong","workable","catastrophic","realtime_hard","no", deterministic=False, regulated_forbidden=False),
    Project("Case-law memo", "Draft legal memos citing real case law",
            "weak","weak","high","interactive","limited", deterministic=False, regulated_forbidden=False),
]
for p in PROJECTS: print(f"- {p.name}: {p.ask}")


- Invoice totals: Automate adding up invoice line items
- Support triage: Route and draft replies for customer support tickets
- Fraud blocking: Auto-decline transactions the model thinks are fraud, no human
- Case-law memo: Draft legal memos citing real case law


## 1. Feasibility — five dimensions, four disqualifiers

Before design, score five dimensions and check four hard disqualifiers. The disqualifiers are *gates*, not scores: if any fires, AI is the wrong tool no matter how strong the rest looks. Note latency is judged against **P95**, not the median — the tail is what breaks a real workflow.


In [2]:
SCORE = {"strong":2,"workable":1,"weak":0,"none":0,
         "low":2,"medium":1,"high":0,"catastrophic":-1,
         "batch":2,"interactive":1,"realtime_hard":0,
         "yes":2,"limited":1,"no":0}

def disqualifiers(p):
    hits = []
    if p.deterministic:                 hits.append("core is deterministic — use rules, not a model")
    if p.data_available == "none":      hits.append("required data does not exist")
    if p.error_cost == "catastrophic" and p.hitl_possible == "no":
                                        hits.append("zero error tolerance with no human in the loop")
    if p.regulated_forbidden:           hits.append("regulation forbids it")
    return hits

def feasibility(p):
    dims = {
        "data": p.data_available, "task_structure": p.task_specifiable,
        "error_tolerance": p.error_cost, "latency(P95)": p.latency_need,
        "human_in_loop": p.hitl_possible,
    }
    scores = {k: SCORE[v] for k,v in dims.items()}
    dq = disqualifiers(p)
    total = sum(scores.values())
    verdict = "DISQUALIFIED" if dq else ("fit" if total >= 6 else "weak fit — revisit")
    return {"project": p.name, **dims, "score": total, "verdict": verdict, "disqualifiers": dq}

import pandas as pd
rows = [feasibility(p) for p in PROJECTS]
print(pd.DataFrame(rows)[["project","score","verdict"]].to_string(index=False))
print()
for r in rows:
    if r["disqualifiers"]: print(f'{r["project"]}: ' + "; ".join(r["disqualifiers"]))


       project  score            verdict
Invoice totals      7       DISQUALIFIED
Support triage      6                fit
Fraud blocking      2       DISQUALIFIED
 Case-law memo      2 weak fit — revisit

Invoice totals: core is deterministic — use rules, not a model
Fraud blocking: zero error tolerance with no human in the loop


**Read it.** *Invoice totals* is disqualified — it's arithmetic, a rule, not a model. *Fraud blocking* is disqualified as framed: catastrophic error cost with **no** human in the loop. Reframe it (flag-for-review instead of auto-decline) and the disqualifier clears — the architect's job is to find that reframing. *Support triage* is a clean fit. *Case-law memo* is a weak fit on data/task structure — the signal to invest in retrieval quality before anything else.


## 2. Classify the problem → the simplest approach that can work

Two axes — **generative vs discriminative**, **retrieval vs reasoning** — then the resolution order: prompt → RAG (only if it needs knowledge outside the model) → fine-tune (only after a measured prompting ceiling) → classical ML (bounded, labelled, latency/cost-constrained).


In [3]:
def classify(p):
    generative = p.name in {"Support triage","Case-law memo"}
    needs_outside_knowledge = p.name in {"Support triage","Case-law memo"}
    bounded_labelled = p.name in {"Fraud blocking"}
    if p.deterministic:
        approach = "none — deterministic rules"
    elif bounded_labelled and p.latency_need == "realtime_hard":
        approach = "classical ML (bounded, labelled, low-latency)"
    elif generative and needs_outside_knowledge:
        approach = "RAG (generative + needs knowledge outside training)"
    elif generative:
        approach = "prompted LLM (try this before anything heavier)"
    else:
        approach = "prompted LLM"
    return {"project": p.name,
            "axis_1": "generative" if generative else "discriminative",
            "axis_2": "retrieval" if needs_outside_knowledge else "reasoning",
            "recommended": approach}

print(pd.DataFrame([classify(p) for p in PROJECTS]).to_string(index=False))


       project         axis_1    axis_2                                         recommended
Invoice totals discriminative reasoning                          none — deterministic rules
Support triage     generative retrieval RAG (generative + needs knowledge outside training)
Fraud blocking discriminative reasoning       classical ML (bounded, labelled, low-latency)
 Case-law memo     generative retrieval RAG (generative + needs knowledge outside training)


Note the discipline: *Fraud blocking* is discriminative, bounded and latency-tight — **classical ML**, not an LLM, even though 'AI' was the ask. *Support triage* earns a model, and because it needs product/account knowledge, **RAG** over a bare prompt.


## 3. Decompose the ask — most 'AI systems' are a bundle

A business request is rarely one task. Break it apart; the pieces that are deterministic shouldn't touch a model.


In [4]:
decomposition = [
    ("detect language",            "classifier (cheap, bounded)",       "low"),
    ("classify ticket intent",    "classifier / prompted LLM",         "low"),
    ("look up account + entitlements","deterministic DB query",         "n/a — no model"),
    ("retrieve relevant policy",  "RAG retrieval",                     "medium"),
    ("draft the reply",           "prompted LLM, grounded on retrieval","high"),
    ("decide auto-send vs human", "business rule + confidence gate",   "n/a — no model"),
]
print(pd.DataFrame(decomposition, columns=["sub-task","approach","criticality"]).to_string(index=False))
print("\n2 of 6 sub-tasks need no model at all — that is the point of decomposing.")


                      sub-task                            approach    criticality
               detect language         classifier (cheap, bounded)            low
        classify ticket intent           classifier / prompted LLM            low
look up account + entitlements              deterministic DB query n/a — no model
      retrieve relevant policy                       RAG retrieval         medium
               draft the reply prompted LLM, grounded on retrieval           high
     decide auto-send vs human     business rule + confidence gate n/a — no model

2 of 6 sub-tasks need no model at all — that is the point of decomposing.


## 4. Meaningful tests — not coverage padding

A test earns its place by asserting a **real behaviour** of the framework, and it carries its reason. We don't test that `SCORE['low']==2` (that's restating a constant); we test the *judgments the framework exists to make*.


In [5]:
def run_checks():
    checks = []
    def check(name, cond, why):
        checks.append((name, "PASS" if cond else "FAIL", why))

    # a deterministic task must be refused — the costliest confusion the framework prevents
    check("deterministic -> disqualified",
          feasibility(PROJECTS[0])["verdict"] == "DISQUALIFIED",
          "arithmetic must not be sent to a model")

    # the fraud case is only disqualified because of the no-human + catastrophic COMBINATION
    fraud = PROJECTS[2]
    check("catastrophic + no HITL -> disqualified",
          "zero error tolerance with no human in the loop" in disqualifiers(fraud),
          "auto-declining with no review is the unsafe part, not the ML itself")
    import dataclasses
    fraud_reviewed = dataclasses.replace(fraud, hitl_possible="yes")
    check("...clears when a human reviews",
          disqualifiers(fraud_reviewed) == [],
          "reframing to flag-for-review is the architect's fix")

    # classification must route a bounded low-latency task to classical ML, not an LLM
    check("bounded+realtime -> classical ML",
          "classical ML" in classify(fraud)["recommended"],
          "an LLM is the wrong tool for a tight-latency bounded classifier")

    # a knowledge-needing generative task must get RAG, not a bare prompt
    check("knowledge-needing generation -> RAG",
          classify(PROJECTS[1])["recommended"].startswith("RAG"),
          "grounding beats hoping the base model knows your policies")

    import pandas as pd
    df = pd.DataFrame(checks, columns=["check","result","why it matters"])
    print(df.to_string(index=False))
    assert all(r=="PASS" for _,r,_ in checks), "a feasibility/classification invariant broke"
    print("\nall", len(checks), "meaningful checks passed")

run_checks()


                                 check result                                                      why it matters
         deterministic -> disqualified   PASS                              arithmetic must not be sent to a model
catastrophic + no HITL -> disqualified   PASS auto-declining with no review is the unsafe part, not the ML itself
        ...clears when a human reviews   PASS                 reframing to flag-for-review is the architect's fix
      bounded+realtime -> classical ML   PASS     an LLM is the wrong tool for a tight-latency bounded classifier
   knowledge-needing generation -> RAG   PASS           grounding beats hoping the base model knows your policies

all 5 meaningful checks passed


Five tests, five real decisions the framework must get right — each with the reason it exists. That is the bar every skill in this factory holds generated code to: a test asserts a behaviour, contract, or safety property and says why; if it only moves a coverage number, it isn't written.


## Check your understanding

**Q1.** *Invoice totals* scored fine on data and task structure but was still rejected. Why?
<details><summary>answer</summary>The four disqualifiers are gates, not scores. A deterministic core (arithmetic) fails outright — a model adds cost, latency and error to something a rule does exactly. Strong scores elsewhere can't buy that back.</details>

**Q2.** *Fraud blocking* was disqualified, then cleared. What single change cleared it, and what does that teach about the architect's role?
<details><summary>answer</summary>Adding a human in the loop (flag-for-review instead of auto-decline). The ML was never the unsafe part — the *autonomy* was. The architect's job is to find the reframing that keeps the value and removes the disqualifier.</details>

**Q3.** Why does *Support triage* get RAG while a different generative task might get just a prompt?
<details><summary>answer</summary>Resolution order: prompt first, add RAG only when the answer needs knowledge outside the model's training. Triage needs current product/account/policy facts, so it crosses into RAG; a task answerable from general capability stays a prompt.</details>

**Q4.** The decomposition sent 2 of 6 sub-tasks to 'no model'. Why is finding those the point?
<details><summary>answer</summary>Every sub-task needlessly routed through a model adds cost, latency, and a new failure mode. Account lookup and the auto-send decision are deterministic; making them deterministic shrinks the blast radius and the bill, and sharpens what the model is actually for.</details>

**Q5.** Why is latency judged on P95 rather than the median?
<details><summary>answer</summary>LLM latency has a long tail. A good median hides the P95/P99 requests that blow the workflow's time budget — and those stack across sequential AI calls. You design and alert on the tail, not the comfortable middle.</details>
